In [1]:
# Importing Modules
import torch
import pandas as pd
import numpy as np
import itertools
from src.dataloader import loadData
from src.model import SimpleGCN, SimpleGAT, SimpleGIN, SimpleGraphSAGE
from src.train_test import TrainGNN, TestGNN
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
######################## DATA-1 ##################################
# Loading ESOL data
esol_train_data = pd.read_csv("data/train/ESOL.csv")
esol_test_data = pd.read_csv("data/test/ESOL.csv")

######################## DATA-2 ##################################
# Loading FreeSolv data
freeSolv_train_data = pd.read_csv("data/train/FreeSolv.csv")
freeSolv_test_data = pd.read_csv("data/test/FreeSolv.csv")

######################## DATA-3 ##################################
# Loading Lipophilicity data
lipophilicity_train_data = pd.read_csv("data/train/Lipophilicity.csv")
lipophilicity_test_data = pd.read_csv("data/test/Lipophilicity.csv")

In [3]:
# Models
model_dict = {
        "SimpleGCN":SimpleGCN, 
		"SimpleGAT":SimpleGAT, 
		"SimpleGIN":SimpleGIN,
		"SimpleGraphSAGE":SimpleGraphSAGE
}

# Models Hyperparameters
model_hyperparams_dict = {
        "SimpleGCN":[0.01, 64, 8],
        "SimpleGAT":[0.005, 128, 32],
        "SimpleGIN":[0.001, 64, 4],
        "SimpleGraphSAGE":[0.005, 128, 4]
}

# Function to run analysis
def RunGNN(model, train_data, test_data, dataName, modelName, params, f):
    
	# Training data and target labels
	smiles_X_train = train_data["smiles"]
	X_train = smiles_X_train.to_numpy()
	y_train = train_data["target"]
	y_train = y_train.to_numpy()

    # Testing data and target labels
	smiles_X_test = test_data["smiles"]
	X_test = smiles_X_test.to_numpy()
	y_test = test_data["target"]
	y_test = y_test.to_numpy()

    # Params
	lr, h_dim, b_size = params

	# Loading dataset
	train_loader = loadData(X_train, y_train, batch_size=b_size, f_in=f)
	test_loader = loadData(X_test, y_test, batch_size=b_size, f_in=f)

	# Model
	model = model_dict[modelName](num_features=f, hidden_dim=h_dim, num_classes=1)

	# Model training
	model = TrainGNN(model, train_loader, epochs=100, learning_rate=lr)

	# Model testing
	y_test, y_pred = TestGNN(model, test_loader)

	# Calculate MAE
	mae = mean_absolute_error(y_test, y_pred)

	# Calculate RMSE
	rmse = root_mean_squared_error(y_test, y_pred)
    
    # Calculate pearson r and p_val
	r, p_val = pearsonr(y_test, y_pred)
    
	# Return results
	return pd.DataFrame({"Data":[dataName], "Model":[modelName],"n_Features":f,
			"MAE":round(mae, 2), "RMSE":[round(rmse, 2)], 
			"r":[round(r, 2)], "p-val":[round(p_val, 3)]})

# Feature Ablation

In [4]:
# Data dict
datasets = {
    "ESOL":{"train":esol_train_data, "test":esol_test_data},
    "FreeSolv":{"train":freeSolv_train_data, "test":freeSolv_test_data},
    "Lipophilicity":{"train":lipophilicity_train_data, "test":lipophilicity_test_data}
}

In [5]:
# List to store results
temp_out = []

# Loop for models
for modelName, model in model_dict.items():
    # Loop for dataset
    for dataName, data in datasets.items():
        # Loop for number of features
        for f in [2, 4, 6, 8]:
            # Run Analysis for model, dataset and number of features
            temp_out.append(RunGNN(model, data["train"], data["test"], dataName, modelName, model_hyperparams_dict[modelName], f))

# Final output
GNN_out = pd.concat(temp_out, ignore_index=True)
GNN_out.to_csv("results/Output_GNN_Feature_Ablation_Analysis.csv")
GNN_out

/tmp/ipykernel_2385829/2184828238.py:55: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p_val = pearsonr(y_test, y_pred)


,Data,Model,n_Features,MAE,RMSE,r,p-val
0,ESOL,SimpleGCN,2,1.67,2.11,0.24,0.0
1,ESOL,SimpleGCN,4,1.45,1.87,0.52,0.0
2,ESOL,SimpleGCN,6,0.99,1.30,0.80,0.0
3,ESOL,SimpleGCN,8,1.02,1.31,0.80,0.0
4,FreeSolv,SimpleGCN,2,2.45,3.47,0.55,0.0
5,FreeSolv,SimpleGCN,4,2.16,2.96,0.70,0.0
6,FreeSolv,SimpleGCN,6,2.56,3.30,0.67,0.0
7,FreeSolv,SimpleGCN,8,2.39,2.98,0.74,0.0
8,Lipophilicity,SimpleGCN,2,0.98,1.22,NaN,NaN
9,Lipophilicity,SimpleGCN,4,0.94,1.18,0.27,0.0


In [6]:
# Function to plot barplot showing result
def plot_bar(data, target):
    sns.set_theme(style="ticks", context="paper")
    g = sns.catplot(data=data, kind="point", x="n_Features", y=target, hue="Model",
                    col="Data", col_wrap=3, sharey=False, height=4, width=0.8,
                    aspect=0.8, dodge=False, legend=True)
    g.set_axis_labels("Number of Features", f"{target}")
    g.set_titles("{col_name}")
    for ax in g.axes.flat:
        ax.tick_params(axis="x", rotation=40)
        ax.grid()
    plt.savefig(f"results/GNN_Analysis_Feature_Ablation_Plot.png", dpi=300)
    plt.close()

In [7]:
# Plotting barplot for RMSE
plot_bar(GNN_out, "RMSE")

# Feature Ablation: Validation

In [8]:
# Data dict
datasets = {
    "Lipophilicity_500":{"train":lipophilicity_train_data.sample(500), "test":lipophilicity_test_data},
    "Lipophilicity_1500":{"train":lipophilicity_train_data.sample(1500), "test":lipophilicity_test_data},
    "Lipophilicity_3000":{"train":lipophilicity_train_data.sample(3000), "test":lipophilicity_test_data}
}

In [9]:
# List to store results
temp_out = []

# Loop for models
for modelName, model in model_dict.items():
    # Loop for dataset
    for dataName, data in datasets.items():
        # Loop for number of features
        for f in [2, 4, 6, 8]:
            # Run Analysis for model, dataset and number of features
            temp_out.append(RunGNN(model, data["train"], data["test"], dataName, modelName, model_hyperparams_dict[modelName], f))

# Final output
GNN_out = pd.concat(temp_out, ignore_index=True)
GNN_out.to_csv("results/Output_Validation_GNN_Feature_Ablation_Analysis.csv")
GNN_out

,Data,Model,n_Features,MAE,RMSE,r,p-val
0,Lipophilicity_500,SimpleGCN,2,0.97,1.21,0.10,0.003
1,Lipophilicity_500,SimpleGCN,4,0.97,1.25,0.14,0.000
2,Lipophilicity_500,SimpleGCN,6,0.94,1.22,0.28,0.000
3,Lipophilicity_500,SimpleGCN,8,0.95,1.17,0.27,0.000
4,Lipophilicity_1500,SimpleGCN,2,0.96,1.19,0.22,0.000
5,Lipophilicity_1500,SimpleGCN,4,0.94,1.18,0.30,0.000
6,Lipophilicity_1500,SimpleGCN,6,0.93,1.15,0.33,0.000
7,Lipophilicity_1500,SimpleGCN,8,0.92,1.15,0.34,0.000
8,Lipophilicity_3000,SimpleGCN,2,0.97,1.20,0.16,0.000
9,Lipophilicity_3000,SimpleGCN,4,0.94,1.17,0.29,0.000


In [10]:
# Function to plot barplot showing result
def plot_bar(data, target):
    sns.set_theme(style="ticks", context="paper")
    g = sns.catplot(data=data, kind="point", x="n_Features", y=target, hue="Model",
                    col="Data", col_wrap=3, sharey=False, height=4, width=0.8,
                    aspect=0.8, dodge=False, legend=True)
    g.set_axis_labels("Number of Features", f"{target}")
    g.set_titles("{col_name}")
    for ax in g.axes.flat:
        ax.tick_params(axis="x", rotation=40)
        ax.grid()
    plt.savefig(f"results/GNN_Analysis_Feature_Ablation_Validation_Plot.png", dpi=300)
    plt.close()

In [11]:
# Plotting barplot for RMSE
plot_bar(GNN_out, "RMSE")